# 01 — Exploratory Data Analysis: NIH Chest X-ray14

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arudaev/chexvision/blob/main/notebooks/01_eda.ipynb)

This notebook explores the dataset before training:
- Dataset size and structure
- Label distribution (class imbalance analysis)
- Sample images per pathology
- Multi-label co-occurrence patterns
- Train/Val/Test split statistics

> **Cloud-only**: Run this notebook on Google Colab. The dataset is ~45GB and streams from HuggingFace.

In [ ]:
# === Colab Setup (skip if running elsewhere) ===
import os
if 'COLAB_GPU' in os.environ or 'google.colab' in str(globals()):
    !git clone https://github.com/arudaev/chexvision.git /content/chexvision 2>/dev/null || true
    %cd /content/chexvision
    !pip install -q -e '.[dev]'
    os.environ['CHEXVISION_ALLOW_LOCAL'] = '1'
    print('Colab environment ready.')
else:
    import sys
    sys.path.insert(0, '..')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter
from datasets import load_dataset

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# Stream dataset from HuggingFace (no local download needed)
ds = load_dataset('alkzar90/NIH-Chest-X-ray-dataset', streaming=True)
print('Available splits:', list(ds.keys()))

# Take a manageable sample for EDA
train_samples = list(ds['train'].take(5000))
print(f'Loaded {len(train_samples)} samples for EDA')
print(f'Sample keys: {list(train_samples[0].keys())}')

In [ ]:
# Build a DataFrame from samples
records = []
for s in train_samples:
    labels = s.get('labels', s.get('label', 'No Finding'))
    records.append({'labels': labels})
df = pd.DataFrame(records)
print(f'Total samples in EDA subset: {len(df):,}')
df.head()

In [ ]:
# Label distribution
PATHOLOGY_LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass',
    'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema',
    'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia',
]

label_counts = Counter()
for labels_str in df['labels']:
    if pd.isna(labels_str) or labels_str == 'No Finding':
        label_counts['No Finding'] += 1
    else:
        for label in str(labels_str).split('|'):
            label_counts[label.strip()] += 1

fig, ax = plt.subplots(figsize=(12, 6))
labels, counts = zip(*sorted(label_counts.items(), key=lambda x: -x[1]))
ax.barh(labels, counts, color='steelblue')
ax.set_xlabel('Count')
ax.set_title('Label Distribution — NIH Chest X-ray14 (EDA sample)')
plt.tight_layout()
plt.show()

In [ ]:
# Binary distribution: Normal vs Abnormal
df['is_abnormal'] = df['labels'].apply(lambda x: 0 if pd.isna(x) or x == 'No Finding' else 1)
print(f'Normal: {(df["is_abnormal"] == 0).sum():,}')
print(f'Abnormal: {(df["is_abnormal"] == 1).sum():,}')
print(f'Abnormal ratio: {df["is_abnormal"].mean():.2%}')

fig, ax = plt.subplots(figsize=(6, 4))
df['is_abnormal'].value_counts().plot.pie(labels=['Normal', 'Abnormal'], autopct='%1.1f%%', ax=ax)
ax.set_title('Normal vs Abnormal Distribution')
ax.set_ylabel('')
plt.show()

In [ ]:
# Multi-label co-occurrence matrix
cooccurrence = np.zeros((len(PATHOLOGY_LABELS), len(PATHOLOGY_LABELS)), dtype=int)

for labels_str in df['labels']:
    if pd.isna(labels_str) or labels_str == 'No Finding':
        continue
    present = [PATHOLOGY_LABELS.index(l.strip()) for l in str(labels_str).split('|') if l.strip() in PATHOLOGY_LABELS]
    for i in present:
        for j in present:
            cooccurrence[i][j] += 1

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cooccurrence, xticklabels=PATHOLOGY_LABELS, yticklabels=PATHOLOGY_LABELS,
            cmap='YlOrRd', annot=True, fmt='d', ax=ax)
ax.set_title('Pathology Co-occurrence Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Sample images (streamed from HF, not stored locally)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
samples_for_viz = train_samples[:10]

for ax, sample in zip(axes.flat, samples_for_viz):
    img = sample['image']
    ax.imshow(img, cmap='gray')
    label_text = str(sample.get('labels', sample.get('label', '?')))[:30]
    ax.set_title(label_text, fontsize=8)
    ax.axis('off')

plt.suptitle('Sample Chest X-ray Images (streamed from HuggingFace)', fontsize=14)
plt.tight_layout()
plt.show()